# ChatModel 和 AgenticModel 区别
    ChatModel 是面向传统“消息对话”的模型接口，而 AgenticModel 是专为“智能体工作流”设计的升级接口。
    两者的核心区别不在于底层大模型本身，而在于消息协议与工具交互方式：
        ChatModel 使用经典的 Message 结构。
        AgenticModel 使用更丰富的 AgenticMessage 结构，以原生支持推理、多模态及更复杂的工具调用。

## 🧩 核心区别对比
        消息与内容结构

            ChatModel：使用传统的 schema.Message。内容通常是纯文本，工具调用与结果往往需要依赖独立的角色（如 tool role）或特定字段来表达，结构相对扁平。
            AgenticModel：使用 schema.AgenticMessage。一条消息由多个有序的 ContentBlock 组成，可以同时承载文本、推理过程（Reasoning）、多模态内容、工具调用及结果等，表达能力更强。

        工具调用机制

            ChatModel：通常通过 WithTools 或已弃用的 BindTools 传入工具定义。工具选择逻辑相对基础。
            AgenticModel：提供了更精细的控制选项。除了基础工具，还支持 WithDeferredTools（延迟加载工具） 和 WithToolSearchTool（工具搜索工具），允许模型在需要时动态检索和加载工具，更适合工具集庞大的场景。

        接口定位与状态

            ChatModel：Eino 中的经典接口。注意，旧版的 ChatModel 接口因使用 BindTools 原地修改实例存在并发风险，已被标记为弃用（Deprecated），推荐使用 ToolCallingChatModel。
            AgenticModel：是 BaseModel[*schema.AgenticMessage] 的类型别名，属于较新的抽象层，专门用于适配智能体路径（Agentic Path），设计上考虑了更复杂的推理与工具编排需求。

## 📌 适用场景怎么选
        选择取决于你的应用是否需要复杂的推理链或大规模动态工具集。

        选 ChatModel（或 ToolCallingChatModel）：常规对话与简单工具调用
        如果你的应用是标准的聊天机器人，或者只需要模型在对话中调用少数几个固定的工具（如查天气、算数学），使用经典的 ChatModel 系列接口就足够了。它的生态更成熟，所有主流模型提供商（OpenAI、Claude、DeepSeek 等）都优先支持这条路径。
        选 AgenticModel：深度推理与复杂智能体
        当你需要模型在响应中保留思维链（Reasoning Content），或者你的智能体需要接入海量工具（需要通过工具搜索来动态选取）时，AgenticModel 是更合适的选择。它的 ContentBlock 结构能更好地保留模型的原生推理过程，而不仅仅是最终文本或工具参数。


# agenticopenai.ChatModel 和 agenticopenai.ResponsesModel 区别
    agenticopenai.ChatModel 和 agenticopenai.ResponsesModel 的核心区别在于它们底层对接的 OpenAI API 端点不同。这直接决定了它们的功能集、响应元数据的结构，以及对高级 Agentic 特性的支持程度。

## 核心区别对比
    底层 API
        ChatModel：对接 Chat Completions API（传统对话补全接口）。
        ResponsesModel：对接 Responses API（OpenAI 较新的、面向 Agent 的接口）。

    响应元数据
        ChatModel：使用通用的 Extension any 字段，你需要将其断言为 *agenticopenai.ChatResponseMetaExtension 才能读取 ID、结束原因等信息。
        ResponsesModel：使用强类型的 OpenAIExtension *openai.ResponseMetaExtension，无需断言即可直接访问服务端状态、响应 ID 等。
    高级缓存与对话延续
        ChatModel：不支持 Responses 特有的缓存延续机制。
        ResponsesModel：支持通过 EnableAutoCache 和 WithHeadPreviousResponseID 等选项，利用上轮响应 ID 来延续多轮对话和前缀缓存，效率更高。
    服务端工具
        ChatModel：主要通过函数调用（Function Tools）机制。
        ResponsesModel：原生支持更丰富的服务端工具（如 Web Search、File Search、Code Interpreter），并提供了专门的选项（如 WithResponsesServerTools）来配置。

    如何选择
        选 ResponsesModel：如果你需要利用 OpenAI 最新的 Agent 能力（特别是服务端内置工具），或者需要精细控制多轮对话的缓存以节省 token 和延迟，它是更合适的选择。
        选 ChatModel：如果你只需要基础的对话和函数调用功能，或者你的代码需要与仅支持传统 Chat Completions 端点的代理/兼容层对接，它依然稳定可用。